# Impact of Prompting Strategies on Information Extraction from AI News Articles Using Open-Source LLMs

In [3]:
import pandas as pd

# Import dataset
df = pd.read_csv("MIT_AI_ARTICLES.csv")

print("Original article number:", len(df))

# Hold on only necessary columns
df = df[[
    "title",
    "publication_date",
    "summary",
    "body",
    "url"
]]

# Remove the ones with an empty Body.
df = df.dropna(subset=["body"])

# Calculate word count
df["word_count"] = df["body"].astype(str).str.split().str.len()

# Remove shorter or longer articles
clean_df = df[
    (df["word_count"] >= 200) &
    (df["word_count"] <= 3000)
]

print("Number of cleaned articles:", len(clean_df))

# Kaydet
clean_df.to_csv("clean_articles.csv", index=False)

clean_df.head()

Original article number: 314
Number of cleaned articles: 314


,title,publication_date,summary,body,url,word_count
0,A new model predicts how molecules will dissol...,"August 19, 2025",Solubility predictions could make it easier to...,"Using machine learning, MIT chemical engineers...",https://news.mit.edu/2025/new-model-predicts-h...,1149
1,Researchers glimpse the inner workings of prot...,"August 18, 2025",A new approach can reveal the features AI mode...,"Within the past few years, models that can pre...",https://news.mit.edu/2025/researchers-glimpse-...,1025
2,How AI could speed the development of RNA vacc...,"August 15, 2025",MIT engineers used a machine-learning model to...,"Using artificial intelligence, MIT researchers...",https://news.mit.edu/2025/how-ai-could-speed-d...,1006
3,"Using generative AI, researchers design compou...","August 14, 2025",The team used two different AI approaches to d...,"With help from artificial intelligence, MIT re...",https://news.mit.edu/2025/using-generative-ai-...,1076
4,A new way to test how well AI systems classify...,"August 13, 2025",As large language models increasingly dominate...,Is this movie review a rave or a pan? Is this ...,https://news.mit.edu/2025/new-way-test-how-wel...,1186


In [13]:
clean_df["word_count"].describe()

count     314.000000
mean     1053.404459
std       283.800000
min       259.000000
25%       910.750000
50%      1051.500000
75%      1172.000000
max      2304.000000
Name: word_count, dtype: float64

After removing missing values and filtering articles outside the 200–3000 word range, all 314 articles remained in the dataset. 
Therefore, no additional article removal was required.

### Control of numbers company, model and tech at all dataset

In [31]:
import re
from collections import Counter

companies = [
    "OpenAI",
    "Google",
    "Meta",
    "Microsoft",
    "Anthropic",
    "Nvidia",
    "Amazon",
    "IBM",
    "Adobe",
    "Tesla",
    "Apple",
    "DeepMind"
]

all_text = " ".join(clean_df["body"].astype(str))

for company in companies:
    count = len(re.findall(company, all_text, flags=re.IGNORECASE))
    print(f"{company}: {count}")

OpenAI: 28
Google: 79
Meta: 108
Microsoft: 14
Anthropic: 6
Nvidia: 21
Amazon: 34
IBM: 107
Adobe: 8
Tesla: 4
Apple: 17
DeepMind: 9


In [33]:
models = [
    "GPT",
    "Claude",
    "Gemini",
    "Llama",
    "Mistral"
]

for model in models:
    print(model, all_text.count(model))

GPT 115
Claude 13
Gemini 5
Llama 4
Mistral 3


In [50]:
techs = [
    "Generative AI",
    "Large Language Model",
    "Machine Learning",
    "Deep Learning",
    "Reinforcement Learning",
    "Neural Network",
    "Computer Vision",
    "Multimodal"
]

for tech in techs:
    print(tech, all_text.count(tech))

Generative AI 24
Large Language Model 0
Machine Learning 42
Deep Learning 2
Reinforcement Learning 0
Neural Network 0
Computer Vision 15
Multimodal 2


### Selecting 100 articles

In [53]:

sample_df = clean_df.sample(
    n=100,
    random_state=42
)

sample_df.to_csv(
    "evaluation_articles.csv",
    index=False
)

print("Selected articles:", len(sample_df))

Selected articles: 100


In [55]:
sample_df[["title","word_count"]].head(10)

,title,word_count
132,Startup’s autonomous drones precisely track wa...,1173
33,Using generative AI to help robots jump higher...,886
222,Melissa Choi named director of MIT Lincoln Lab...,1350
57,MIT announces the Initiative for New Manufactu...,1041
256,Julie Shah named head of the Department of Aer...,649
204,Dimitris Bertsimas named vice provost for open...,917
25,AI shapes autonomous underwater “gliders”,1096
9,MIT tool visualizes and edits “physically impo...,920
242,A community collaboration for progress,1062
224,Eric Evans receives Department of Defense Meda...,497


In [57]:
sample_text = " ".join(sample_df["body"].astype(str))

companies = [
    "OpenAI",
    "Anthropic",
    "Google",
    "Meta",
    "Microsoft",
    "Nvidia"
]

for company in companies:
    print(company, sample_text.count(company))

OpenAI 6
Anthropic 0
Google 16
Meta 6
Microsoft 2
Nvidia 0


In [59]:
keywords = [
    "OpenAI",
    "Google",
    "Meta",
    "Microsoft",
    "Anthropic",
    "Nvidia",
    "GPT",
    "Claude",
    "Gemini",
    "Llama"
]

mask = clean_df["body"].astype(str).apply(
    lambda x: any(k.lower() in x.lower() for k in keywords)
)

entity_articles = clean_df[mask]

print(len(entity_articles))

134


In [61]:
sample_df = entity_articles.sample(
    n=100,
    random_state=42
)

To ensure meaningful evaluation, articles containing at least one target entity (company name) were first identified. A random sample of 100 articles was then selected from this subset.